In [14]:
import cv2
import numpy as np
import pyttsx3

In [15]:
# Voice Engine

engine = pyttsx3.init()
engine.setProperty('rate', 150)

def speak(text):
    engine.say(text)
    engine.runAndWait()

In [16]:
# Fire Detection (HSV)

def detect_fire_mask(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    lower1 = np.array([0, 120, 200])
    upper1 = np.array([35, 255, 255])

    lower2 = np.array([35, 50, 200])
    upper2 = np.array([60, 255, 255])

    mask1 = cv2.inRange(hsv, lower1, upper1)
    mask2 = cv2.inRange(hsv, lower2, upper2)

    mask = cv2.bitwise_or(mask1, mask2)

    kernel = np.ones((5,5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_DILATE, kernel)

    return mask

In [ ]:
# Intensity Function

def calculate_intensity(frame, prev_frame=None):
    mask = detect_fire_mask(frame)

    fire_pixels = np.sum(mask > 0)
    total_pixels = frame.shape[0] * frame.shape[1]
    area_ratio = fire_pixels / total_pixels

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    brightness = np.mean(gray[mask > 0]) if fire_pixels > 0 else 0
    brightness_norm = brightness / 255

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)

    hot_pixels = np.sum((h < 35) & (v > 200) & (mask > 0))
    hot_ratio = hot_pixels / fire_pixels if fire_pixels > 0 else 0

    flicker = 0
    if prev_frame is not None:
        diff = cv2.absdiff(frame, prev_frame)
        diff_gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
        flicker = np.mean(diff_gray[mask > 0]) / 255 if fire_pixels > 0 else 0

    intensity = (
        0.5 * area_ratio +
        0.2 * brightness_norm +
        0.2 * hot_ratio +
        0.1 * flicker
    )

    return intensity, mask


In [ ]:

# Webcam Setup

cap = cv2.VideoCapture(0)

frame1 = None
frame2 = None

stage = 0  # 0 = waiting for first capture, 1 = waiting for second

print("Press 'c' to capture fire images step-by-step")
print("Press 'r' to reset")
print("Press 'q' to quit")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    display = frame.copy()

    # Show status on screen
    if stage == 0:
        cv2.putText(display, "Capture FIRST fire (press c)", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
    elif stage == 1:
        cv2.putText(display, "Capture SECOND fire (press c)", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)

    cv2.imshow("Live Feed", display)

    key = cv2.waitKey(1)

    
    # CAPTURE LOGIC
    
    if key == ord('c'):
        if stage == 0:
            frame1 = frame.copy()
            stage = 1
            print("First fire captured and locked.")
            speak("First fire captured.")

            cv2.imshow("Fire 1", frame1)

        elif stage == 1:
            frame2 = frame.copy()
            print("Second fire captured.")
            speak("Second fire captured.")

            cv2.imshow("Fire 2", frame2)

            
            # INTENSITY COMPARISON
            
            i1, mask1 = calculate_intensity(frame1)
            i2, mask2 = calculate_intensity(frame2, frame1)

            print(f"Fire 1 Intensity: {i1:.3f}")
            print(f"Fire 2 Intensity: {i2:.3f}")

            cv2.imshow("Mask 1", mask1)
            cv2.imshow("Mask 2", mask2)

            if i1 > i2:
                msg = "Warning! First fire is more intense. Evacuate immediately, take route 2."
            elif i2 > i1:
                msg = "Warning! Second fire is more intense. Evacuate immediately, take route 1"
            else:
                msg = "Both fires have similar intensity."

            print(msg)
            speak(msg)

            # Reset automatically after comparison
            stage = 0

    
    # RESET
    
    elif key == ord('r'):
        frame1 = None
        frame2 = None
        stage = 0
        print("System reset.")
        speak("System reset.")

    elif key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Press 'c' to capture fire images step-by-step
Press 'r' to reset
Press 'q' to quit
First fire captured and locked.
Second fire captured.
Fire 1 Intensity: 0.307
Fire 2 Intensity: 0.000
Warning! First fire is more intense. Evacuate immediately, take route 2.
